# 00 — Data Ingestion Pipeline (v1)

**Purpose:** Reproducible, idempotent ingestion of all raw OHLCV data.

| Layer | Assets | Interval | Destination |
|-------|--------|----------|-------------|
| Primary | 10 USDT pairs | 1 h | `data/raw/{SYMBOL}_1h.parquet` |
| Extended | BTC only | 5 m | `data/raw/BTCUSDT_5m.parquet` |

**Design guarantees:**
- **Idempotent:** safe to re-run; picks up where the last save left off.
- **Incremental:** only fetches the delta since the latest stored candle.
- **Gap-aware:** detects exchange down-time gaps and forward-fills them.
- **Schema-pinned:** explicit `float32` OHLCV, UTC `open_time` index.


In [1]:
from __future__ import annotations

import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}  |  python {sys.version.split()[0]}")


pandas 3.0.2  |  numpy 2.4.4  |  python 3.13.12


In [2]:
# ── Repo root (walk up until pyproject.toml is found) ─────────────────────
def _find_repo_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    raise RuntimeError("pyproject.toml not found; run from within the repo")

REPO_ROOT = _find_repo_root()
RAW_DIR   = REPO_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"RAW_DIR   : {RAW_DIR}")


REPO_ROOT : /Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system
RAW_DIR   : /Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system/data/raw


In [3]:
# ── Asset universe ─────────────────────────────────────────────────────────
SYMBOLS_1H: list[str] = [
    "BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "SOLUSDT",
    "ADAUSDT", "DOGEUSDT", "AVAXUSDT", "DOTUSDT", "LINKUSDT",
]
SYMBOL_5M  = "BTCUSDT"

# ── Interval metadata ──────────────────────────────────────────────────────
INTERVAL_1H = "1h"
INTERVAL_5M = "5m"
INTERVAL_MINUTES: dict[str, int] = {"1h": 60, "5m": 5}

# ── Binance public REST ────────────────────────────────────────────────────
BINANCE_KLINES = "https://api.binance.com/api/v3/klines"
REQUEST_LIMIT  = 1000    # max rows per call (Binance hard cap)
REQUEST_SLEEP  = 0.15    # seconds between paginated calls

# ── Column schema (float32 for compact storage) ────────────────────────────
OHLCV_COLS  = ["open", "high", "low", "close", "volume"]
OHLCV_DTYPE = {c: "float32" for c in OHLCV_COLS}

print(f"Universe  : {SYMBOLS_1H}")
print(f"5-min BTC : {SYMBOL_5M}")
print(f"OHLCV dtype: float32 (saves ~50 % vs float64)")


Universe  : ['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'XRPUSDT', 'SOLUSDT', 'ADAUSDT', 'DOGEUSDT', 'AVAXUSDT', 'DOTUSDT', 'LINKUSDT']
5-min BTC : BTCUSDT
OHLCV dtype: float32 (saves ~50 % vs float64)


In [4]:
# ── Time conversion helpers ────────────────────────────────────────────────

def _ms_to_ts(ms: int) -> pd.Timestamp:
    return pd.Timestamp(ms, unit="ms", tz="UTC")

def _ts_to_ms(ts) -> int:
    if isinstance(ts, pd.Timestamp):
        return int(ts.value // 1_000_000)
    if isinstance(ts, datetime):
        return int(ts.timestamp() * 1000)
    raise TypeError(f"Cannot convert {type(ts)} to ms")

def _now_ms() -> int:
    return int(datetime.now(timezone.utc).timestamp() * 1000)


# ── Binance klines fetcher (paginated) ────────────────────────────────────

def fetch_klines(
    symbol: str,
    interval: str,
    start_ms: int,
    end_ms: int,
    limit: int = REQUEST_LIMIT,
    sleep_s: float = REQUEST_SLEEP,
) -> pd.DataFrame:
    """Paginated Binance klines fetch → UTC-indexed OHLCV DataFrame."""
    rows: list = []
    cursor   = start_ms
    page_n   = 0
    interval_ms = INTERVAL_MINUTES[interval] * 60_000

    while cursor < end_ms:
        params = {
            "symbol":    symbol,
            "interval":  interval,
            "startTime": cursor,
            "endTime":   end_ms,
            "limit":     limit,
        }
        resp = requests.get(BINANCE_KLINES, params=params, timeout=30)
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break

        rows.extend(batch)
        last_open_ms = int(batch[-1][0])
        page_n += 1

        # Progress tick every 25 pages (useful for large 5m pulls)
        if page_n % 25 == 0:
            pct = min(100.0, (last_open_ms - start_ms) / max(1, end_ms - start_ms) * 100)
            dt  = _ms_to_ts(last_open_ms).strftime("%Y-%m-%d")
            print(f"    page {page_n:>5} | {pct:>5.1f}% | up to {dt}", end="\r", flush=True)

        if len(batch) < limit:
            break  # final page reached
        cursor = last_open_ms + interval_ms
        time.sleep(sleep_s)

    if page_n >= 25:
        print()  # newline after \r progress line

    if not rows:
        return pd.DataFrame(columns=OHLCV_COLS)

    df = pd.DataFrame(
        rows,
        columns=[
            "open_time", "open", "high", "low", "close", "volume",
            "close_time", "quote_vol", "n_trades",
            "taker_base", "taker_quote", "_ignore",
        ],
    )[["open_time"] + OHLCV_COLS]

    df["open_time"] = pd.to_datetime(df["open_time"].astype("int64"), unit="ms", utc=True)
    df = df.set_index("open_time").astype(OHLCV_DTYPE)
    return df


# ── Parquet I/O ────────────────────────────────────────────────────────────

def parquet_path(symbol: str, interval: str) -> Path:
    return RAW_DIR / f"{symbol}_{interval}.parquet"

def load_parquet(symbol: str, interval: str) -> Optional[pd.DataFrame]:
    p = parquet_path(symbol, interval)
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    df.index.name = "open_time"
    return df

def save_parquet(df: pd.DataFrame, symbol: str, interval: str) -> None:
    df = df.astype(OHLCV_DTYPE)
    df.to_parquet(parquet_path(symbol, interval))


# ── Earliest available bar on Binance ─────────────────────────────────────

def earliest_available_ms(symbol: str, interval: str) -> int:
    """Return the open_time (ms) of the very first bar on Binance."""
    resp = requests.get(
        BINANCE_KLINES,
        params={"symbol": symbol, "interval": interval,
                "startTime": 0, "limit": 1},
        timeout=15,
    )
    resp.raise_for_status()
    return int(resp.json()[0][0])


print("Helper functions defined.")


Helper functions defined.


In [5]:
# ── Idempotent incremental ingest ──────────────────────────────────────────

def ingest(symbol: str, interval: str) -> dict:
    """
    Fetch and store raw OHLCV for one (symbol, interval) pair.

    Logic:
    - If local parquet exists  → delta fetch from (last_candle + 1 bar) to now.
    - If no local parquet      → full backfill from exchange earliest.
    - After merge: deduplicate (keep last), sort, validate, save.

    Returns a status dict with diagnostics for logging.
    """
    interval_ms = INTERVAL_MINUTES[interval] * 60_000
    now_ms      = _now_ms()

    existing = load_parquet(symbol, interval)

    if existing is not None and len(existing) > 0:
        last_ts  = existing.index[-1]
        start_ms = _ts_to_ms(last_ts) + interval_ms
        mode     = "incremental"
        n_before = len(existing)
    else:
        start_ms = earliest_available_ms(symbol, interval)
        mode     = "full backfill"
        n_before = 0
        existing = None

    # Already up to date?
    if start_ms > now_ms:
        df = load_parquet(symbol, interval)
        return {
            "symbol": symbol, "interval": interval, "mode": "up-to-date",
            "fetched": 0, "n_before": n_before, "n_after": n_before,
            "first": df.index[0], "last": df.index[-1],
        }

    delta   = fetch_klines(symbol, interval, start_ms, now_ms)
    n_fetch = len(delta)

    if existing is not None and n_fetch > 0:
        combined = pd.concat([existing, delta])
    elif n_fetch > 0:
        combined = delta
    else:
        combined = existing if existing is not None else pd.DataFrame(columns=OHLCV_COLS)

    # Deduplicate (keep most-recent copy) and sort
    combined = (
        combined[~combined.index.duplicated(keep="last")]
        .sort_index()
        .astype(OHLCV_DTYPE)
    )

    save_parquet(combined, symbol, interval)

    return {
        "symbol":   symbol,
        "interval": interval,
        "mode":     mode,
        "fetched":  n_fetch,
        "n_before": n_before,
        "n_after":  len(combined),
        "first":    combined.index[0],
        "last":     combined.index[-1],
    }

print("ingest() defined.")


ingest() defined.


## Pre-Run Disk Audit

In [6]:
print("=" * 70)
print("PRE-RUN DISK STATE")
print("=" * 70)

def _audit_row(symbol: str, interval: str) -> dict:
    p  = parquet_path(symbol, interval)
    df = load_parquet(symbol, interval)
    if df is not None and len(df) > 0:
        return {
            "file":    p.name,
            "rows":    len(df),
            "first":   df.index[0].strftime("%Y-%m-%d"),
            "last":    df.index[-1].strftime("%Y-%m-%d"),
            "size_mb": round(p.stat().st_size / 1e6, 2),
            "status":  "EXISTS",
        }
    return {
        "file": p.name, "rows": 0, "first": "—", "last": "—",
        "size_mb": 0.0, "status": "MISSING",
    }

pre_rows = [_audit_row(s, INTERVAL_1H) for s in SYMBOLS_1H]
pre_rows.append(_audit_row(SYMBOL_5M, INTERVAL_5M))
pre_df = pd.DataFrame(pre_rows)
print(pre_df.to_string(index=False))
print()
n_exist = (pre_df["status"] == "EXISTS").sum()
print(f"Files already on disk : {n_exist} / {len(pre_rows)}")
total_existing_mb = pre_df["size_mb"].sum()
print(f"Current total size    : {total_existing_mb:.2f} MB")


PRE-RUN DISK STATE
               file  rows      first       last  size_mb  status
 BTCUSDT_1h.parquet 76525 2017-08-17 2026-05-16     3.45  EXISTS
 ETHUSDT_1h.parquet 76525 2017-08-17 2026-05-16     3.07  EXISTS
 BNBUSDT_1h.parquet 74588 2017-11-06 2026-05-16     2.70  EXISTS
 XRPUSDT_1h.parquet 70321 2018-05-04 2026-05-16     2.48  EXISTS
 SOLUSDT_1h.parquet 50471 2020-08-11 2026-05-16     1.66  EXISTS
 ADAUSDT_1h.parquet 70733 2018-04-17 2026-05-16     2.32  EXISTS
DOGEUSDT_1h.parquet 60113 2019-07-05 2026-05-16     2.33  EXISTS
AVAXUSDT_1h.parquet 49463 2020-09-22 2026-05-16     1.43  EXISTS
 DOTUSDT_1h.parquet 50286 2020-08-18 2026-05-16     1.52  EXISTS
LINKUSDT_1h.parquet 64179 2019-01-16 2026-05-16     2.08  EXISTS
 BTCUSDT_5m.parquet     0          —          —     0.00 MISSING

Files already on disk : 10 / 11
Current total size    : 23.04 MB


## Phase 1 — 1-Hour Ingestion (All 10 Symbols)

Incremental update: fetches only the bars missing since the last saved candle.
First run performs a full backfill from the earliest available bar on Binance.


In [7]:
print("=" * 70)
print("PHASE 1 — 1H INGESTION")
print("=" * 70)

results_1h: list[dict] = []

for sym in SYMBOLS_1H:
    print(f"\n  [{sym}] ...", end=" ", flush=True)
    result = ingest(sym, INTERVAL_1H)
    results_1h.append(result)
    print(
        f"{result['mode']:>15}  |  "
        f"fetched: {result['fetched']:>6}  |  "
        f"total: {result['n_after']:>6}  |  "
        f"{str(result['first'])[:10]} → {str(result['last'])[:10]}"
    )

print("\n✓  Phase 1 complete.")
print(f"   Total new bars fetched : {sum(r['fetched'] for r in results_1h):,}")


PHASE 1 — 1H INGESTION

  [BTCUSDT] ...     incremental  |  fetched:    285  |  total:  76810  |  2017-08-17 → 2026-05-27

  [ETHUSDT] ...     incremental  |  fetched:    285  |  total:  76810  |  2017-08-17 → 2026-05-27

  [BNBUSDT] ...     incremental  |  fetched:    285  |  total:  74873  |  2017-11-06 → 2026-05-27

  [XRPUSDT] ...     incremental  |  fetched:    285  |  total:  70606  |  2018-05-04 → 2026-05-27

  [SOLUSDT] ...     incremental  |  fetched:    285  |  total:  50756  |  2020-08-11 → 2026-05-27

  [ADAUSDT] ...     incremental  |  fetched:    285  |  total:  71018  |  2018-04-17 → 2026-05-27

  [DOGEUSDT] ...     incremental  |  fetched:    285  |  total:  60398  |  2019-07-05 → 2026-05-27

  [AVAXUSDT] ...     incremental  |  fetched:    285  |  total:  49748  |  2020-09-22 → 2026-05-27

  [DOTUSDT] ...     incremental  |  fetched:    285  |  total:  50571  |  2020-08-18 → 2026-05-27

  [LINKUSDT] ...     incremental  |  fetched:    285  |  total:  64464  |  2019-01-

## Phase 2 — 5-Minute Ingestion (BTCUSDT Only)

Maximum-availability historical fetch: paginate from the exchange's earliest
available 5-minute bar to the current minute.

> **Note:** A full backfill (~420,000 bars from 2017) takes ≈ 3–4 minutes.
> Subsequent incremental runs complete in seconds.


In [8]:
print("=" * 70)
print("PHASE 2 — 5M INGESTION (BTCUSDT)")
print("=" * 70)
print(f"\n  [{SYMBOL_5M}/{INTERVAL_5M}] starting (full backfill may take a few minutes)...")

result_5m = ingest(SYMBOL_5M, INTERVAL_5M)

print(
    f"\n  [{SYMBOL_5M}/{INTERVAL_5M}] {result_5m['mode']:>15}  |  "
    f"fetched: {result_5m['fetched']:>7}  |  "
    f"total: {result_5m['n_after']:>7}  |  "
    f"{str(result_5m['first'])[:10]} → {str(result_5m['last'])[:10]}"
)
print("\n✓  Phase 2 complete.")


PHASE 2 — 5M INGESTION (BTCUSDT)

  [BTCUSDT/5m] starting (full backfill may take a few minutes)...
    page   900 |  97.7% | up to 2026-03-14

  [BTCUSDT/5m]   full backfill  |  fetched:  921533  |  total:  921533  |  2017-08-17 → 2026-05-27

✓  Phase 2 complete.


## Continuity & Quality Validation

For each dataset:
1. Assert **no duplicate timestamps**.
2. Detect **gaps** (intervals wider than expected).
3. **Forward-fill** any gaps (OHLC → last known price, volume → 0).
4. Assert **no NaN values** remain after gap fill.


In [9]:
print("=" * 70)
print("CONTINUITY & QUALITY VALIDATION")
print("=" * 70)

def validate_and_heal(symbol: str, interval: str) -> dict:
    """Check duplicates, detect/fill gaps, assert cleanliness."""
    df = load_parquet(symbol, interval)
    if df is None or len(df) == 0:
        return {
            "symbol": symbol, "interval": interval,
            "rows": 0, "duplicates": 0, "gaps": 0,
            "nan_after": 0, "status": "MISSING",
        }

    interval_td = pd.Timedelta(minutes=INTERVAL_MINUTES[interval])

    # ── Duplicates ────────────────────────────────────────────────────────
    n_dupes = int(df.index.duplicated().sum())
    if n_dupes:
        df = df[~df.index.duplicated(keep="last")].sort_index()

    # ── Gap detection ─────────────────────────────────────────────────────
    diffs  = df.index.to_series().diff().dropna()
    gaps   = diffs[diffs > interval_td * 1.5]
    n_gaps = len(gaps)

    if n_gaps:
        full_idx = pd.date_range(
            start=df.index[0], end=df.index[-1],
            freq=interval_td, tz="UTC", name="open_time",
        )
        df = df.reindex(full_idx)
        # Forward-fill OHLC (carry last known price), zero-fill volume
        df[["open", "high", "low", "close"]] = (
            df[["open", "high", "low", "close"]].ffill()
        )
        df["volume"] = df["volume"].fillna(0.0)
        df = df.astype(OHLCV_DTYPE)
        save_parquet(df, symbol, interval)
        print(f"  [{symbol}/{interval}] healed {n_gaps} gap(s) → re-saved.")

    n_nan = int(df.isna().sum().sum())

    return {
        "symbol":    symbol,
        "interval":  interval,
        "rows":      len(df),
        "duplicates": n_dupes,
        "gaps":      n_gaps,
        "nan_after": n_nan,
        "first":     df.index[0].strftime("%Y-%m-%d %H:%M UTC"),
        "last":      df.index[-1].strftime("%Y-%m-%d %H:%M UTC"),
        "status":    "OK" if (n_dupes == 0 and n_nan == 0) else "WARN",
    }

checks: list[dict] = []
for sym in SYMBOLS_1H:
    checks.append(validate_and_heal(sym, INTERVAL_1H))
checks.append(validate_and_heal(SYMBOL_5M, INTERVAL_5M))

check_df = pd.DataFrame(checks)
print()
print(check_df[["symbol","interval","rows","duplicates","gaps","nan_after","status"]].to_string(index=False))

# ── Hard assertions ───────────────────────────────────────────────────────
print()
assert (check_df["duplicates"] == 0).all(), "FAIL: duplicate timestamps detected"
assert (check_df["nan_after"]  == 0).all(), "FAIL: NaN values remain after gap fill"
print("✓  No duplicates.")
print("✓  No NaN values.")
print("✓  All continuity assertions passed.")


CONTINUITY & QUALITY VALIDATION
  [BTCUSDT/1h] healed 28 gap(s) → re-saved.
  [ETHUSDT/1h] healed 28 gap(s) → re-saved.
  [BNBUSDT/1h] healed 27 gap(s) → re-saved.
  [XRPUSDT/1h] healed 25 gap(s) → re-saved.
  [SOLUSDT/1h] healed 10 gap(s) → re-saved.
  [ADAUSDT/1h] healed 25 gap(s) → re-saved.
  [DOGEUSDT/1h] healed 18 gap(s) → re-saved.
  [AVAXUSDT/1h] healed 10 gap(s) → re-saved.
  [DOTUSDT/1h] healed 10 gap(s) → re-saved.
  [LINKUSDT/1h] healed 20 gap(s) → re-saved.
  [BTCUSDT/5m] healed 34 gap(s) → re-saved.

  symbol interval   rows  duplicates  gaps  nan_after status
 BTCUSDT       1h  76938           0    28          0     OK
 ETHUSDT       1h  76938           0    28          0     OK
 BNBUSDT       1h  74995           0    27          0     OK
 XRPUSDT       1h  70694           0    25          0     OK
 SOLUSDT       1h  50776           0    10          0     OK
 ADAUSDT       1h  71106           0    25          0     OK
DOGEUSDT       1h  60442           0    18          0

## Final Storage Summary

In [10]:
print("=" * 70)
print("FINAL STORAGE FOOTPRINT")
print("=" * 70)

summary_rows: list[dict] = []
for sym in SYMBOLS_1H:
    p  = parquet_path(sym, INTERVAL_1H)
    df = load_parquet(sym, INTERVAL_1H)
    if df is None:
        continue
    span = df.index[-1] - df.index[0]
    summary_rows.append({
        "file":      p.name,
        "rows":      f"{len(df):>8,}",
        "from":      df.index[0].strftime("%Y-%m-%d"),
        "to":        df.index[-1].strftime("%Y-%m-%d"),
        "span_days": span.days,
        "size_mb":   round(p.stat().st_size / 1e6, 2),
    })

p5  = parquet_path(SYMBOL_5M, INTERVAL_5M)
df5 = load_parquet(SYMBOL_5M, INTERVAL_5M)
if df5 is not None:
    span5 = df5.index[-1] - df5.index[0]
    summary_rows.append({
        "file":      p5.name,
        "rows":      f"{len(df5):>8,}",
        "from":      df5.index[0].strftime("%Y-%m-%d"),
        "to":        df5.index[-1].strftime("%Y-%m-%d"),
        "span_days": span5.days,
        "size_mb":   round(p5.stat().st_size / 1e6, 2),
    })

sum_df = pd.DataFrame(summary_rows)
print(sum_df.to_string(index=False))

total_mb   = sum(r["size_mb"] for r in summary_rows)
total_rows = sum(int(r["rows"].replace(",", "").strip()) for r in summary_rows)

print()
print(f"{'─' * 70}")
print(f"  Files      : {len(summary_rows)}")
print(f"  Total rows : {total_rows:,}")
print(f"  Total size : {total_mb:.2f} MB")
print(f"  Location   : {RAW_DIR}")
print(f"{'─' * 70}")
print()
print("Pipeline complete. Data is ready for feature engineering.")


FINAL STORAGE FOOTPRINT
               file     rows       from         to  span_days  size_mb
 BTCUSDT_1h.parquet   76,938 2017-08-17 2026-05-27       3205     2.86
 ETHUSDT_1h.parquet   76,938 2017-08-17 2026-05-27       3205     2.68
 BNBUSDT_1h.parquet   74,995 2017-11-06 2026-05-27       3124     2.33
 XRPUSDT_1h.parquet   70,694 2018-05-04 2026-05-27       2945     2.05
 SOLUSDT_1h.parquet   50,776 2020-08-11 2026-05-27       2115     1.48
 ADAUSDT_1h.parquet   71,106 2018-04-17 2026-05-27       2962     1.92
DOGEUSDT_1h.parquet   60,442 2019-07-05 2026-05-27       2518     1.77
AVAXUSDT_1h.parquet   49,768 2020-09-22 2026-05-27       2073     1.29
 DOTUSDT_1h.parquet   50,591 2020-08-18 2026-05-27       2107     1.36
LINKUSDT_1h.parquet   64,524 2019-01-16 2026-05-27       2688     1.83
 BTCUSDT_5m.parquet  923,248 2017-08-17 2026-05-27       3205    26.28

──────────────────────────────────────────────────────────────────────
  Files      : 11
  Total rows : 1,570,020
  Total s